# v3 — Post-hoc division detection (local eval)

**Goal: earn the untouched `0.1 * division_jaccard` term.** The two-pass Hungarian linker is strictly
1:1, so every node has out-degree ≤ 1 → the graph can *never* contain a division (out-degree ≥ 2) →
`division_jaccard = 0` by construction. This notebook uses the **v2 LB-best pipeline** (v2 detection —
HM_THR 0.3, min_dist 3, **no NMS** — + v2 LB-proven linking) and adds a **rule-based post-hoc division
step** *after* linking:

**Why v2 detection and not v2.5b here.** The prior v3 eval rode on v2.5b high-recall detection (HM_THR 0.15
+ NMS). That detection LB-regressed (v2.5b 0.786 vs v2 0.827 under identical linking), AND it floods dense
fields with persistent orphans → the "opposite-side persistent orphan" pattern is pure noise, so division-J
was ≈0.0024 (~400:1 FP). Moving divisions onto **v2's sparser detection** does two things: rides the LB-best
base, and leaves fewer orphans → a cleaner division signature (the hypothesis this notebook now tests).

For each mother `M(t)` that already has exactly one daughter `D1(t+1)`, look for a **second daughter** `D2`
among the orphan births at `t+1` (nodes with no incoming edge) and add the edge `M→D2` when it passes:
- **distance** `‖D2−M‖ ≤ DIV_RADIUS_UM` (=10 µm, ~p90 of GT mother→daughter displacement 9.07; note this is
  *wider* than the 8 µm motion gate on purpose — dividing daughters spread farther than normal motion, so
  the second daughter stays an unlinked orphan we can grab),
- **symmetry** angle between `(D1−M)` and `(D2−M)` `≥ MIN_ANGLE_DEG` (=75°; GT split angle median 138°,
  p10 88° → real daughters sit on opposite sides; a same-side orphan is rejected),
- **persistence** `D2` heads a forward chain of length `≥ MIN_DAUGHTER_LEN` (over-detection orphans die
  out; real daughters continue) — the main noise filter,
- optional **mother persistence** `≥ MIN_MOTHER_LEN` (M is an established track, not noise).
One division per mother; the closest qualifying `D2` wins.

**Why this is low-risk for the main edge score:** the second daughter is almost always an over-detected
*orphan* (unmatched to GT), so the added `M→D2` edge has an unmatched endpoint and is **not** charged as an
edge FP. Loop A below confirms this directly (edge-J OFF vs ON).

## What this eval reports
- **Loop A (edge-J safety):** same last-`EVAL_N_VAL` val as v2/v2.5 → edge-Jaccard **with divisions OFF vs
  ON**. These should be ~equal (proves the division step doesn't dent the 0.827 main score).
- **Loop B (division signal):** the `DIV_RICH_N` train samples with the **most GT divisions** → where
  `division_jaccard` (TP/FP/FN, ±1 frame tol, 7 µm gate) is actually measurable, plus the added-division
  count and edge-J. Tells us whether the detector finds *real* divisions.

**Ground truth on divisions (from util_inspect_data):** 151 GT divisions over 199 train samples, only
**0.117% of edges**, and **84% live in the `6bba` specimen** (125 vs 26 for 44b6). So the ceiling on the
0.1 term is small — treat this as free upside, not a main lever. Both loops are 6bba-heavy (correct, since
that is where divisions are).

**Baselines:** v2 edge-J = **0.859** (LB **0.827**, current best) with division-J = **0**. Any division-J > 0
that doesn't lower edge-J is free LB (up to +0.1 * div-J). Local div-J is a rough proxy — the LB divisions
live on the unseen hidden set, so a good local number is *permission to submit-test*, not proof.

**Setup:** add competition data + `v1_UNet_best.pt`. GPU ON, Internet ON (installs zarr). Run All.

In [9]:
# --- deps ---
import subprocess, sys
try:
    import zarr; assert int(zarr.__version__.split('.')[0]) >= 3
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zarr>=3']); import zarr

import os, glob, time
from collections import defaultdict
from dataclasses import dataclass
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
print('zarr', zarr.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

zarr 3.2.1 | torch 2.10.0+cu128 | cuda True


In [ ]:
# ================= CONFIG =================
INPUT     = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TRAIN_DIR = os.path.join(INPUT, 'train')
VOXEL = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um/voxel (anisotropic)
SCALE = VOXEL
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- UNet architecture: MUST match v1_UNet_best.pt ---
BASE    = 16
STRIDES = ((1,2,2), (1,2,2), (2,2,2), (2,2,2))

# --- detection: v2 preset (FROZEN = LB-BEST 0.827) ---
# v2.5b (HM_THR 0.15 + min_dist 1 + NMS 3.0) LB-regressed to 0.786 vs v2's 0.827 with identical linking
# (local tie 0.8588) -> the high-recall detection density is the LB cost. Reverted to v2 detection here so
# divisions ride on the LB-best base: HM_THR 0.3, min_dist 3, NMS OFF.
NORM_PCT   = (50.0, 99.8)
HM_THR     = 0.3
MIN_DIST   = 3
MAX_PEAKS  = 200000
NMS_UM     = 0.0              # OFF (v2 had no physical NMS; 0.0 -> physical_nms is a no-op)
REFINE     = True
REFINE_WIN = (1, 3, 3)

# --- linking + post-processing: v2 LB-proven (loose 8 / filter 4) ---
TIGHT_UM   = 6.0
LOOSE_UM   = 8.0
VEL_BLEND  = 0.5
CLOSE_GAPS = True
MAX_GAP    = 1
GAP_DIST_UM= 6.0
FILTER_MIN_LEN = 4
PRUNE_ISOLATED = True

# --- NEW: post-hoc division detection (the ONLY variable under test) ---
# Gates set from util_inspect_data stats over 151 GT divisions (199 train samples, 84% in 6bba specimen):
#   mother->daughter displacement um: median 5.75, p90 9.07, max 13.53
#   split angle deg:                   median 138, p10 88.1, p90 165
# NOTE: DIV_RADIUS_UM is intentionally > LOOSE_UM (8): dividing daughters spread far wider than normal
#   motion (median link 2.88 um), so the SECOND daughter sits beyond the motion gate -> stays an orphan
#   we can pick up here. Set radius to ~p90 to catch most second daughters.
# HYPOTHESIS for v2 detection: sparser peaks (HM_THR 0.3, no NMS) leave FEWER persistent orphans than the
#   v2.5b high-recall detection did, so "opposite-side persistent orphan" should be a less noisy division
#   signature -> division precision (div-J) may improve vs the ~0.0024 seen on v2.5b detection.
DIV_ENABLE       = True
DIV_RADIUS_UM    = 10.0   # max mother->second-daughter distance (~p90 9.07 of GT mother-daughter disp)
MIN_ANGLE_DEG    = 75.0   # min angle between (D1-M) and (D2-M); GT p10 = 88 deg -> 75 keeps margin, rejects same-side
MIN_DAUGHTER_LEN = 3      # second daughter must head a forward chain of >= this many nodes (main precision lever)
MIN_MOTHER_LEN   = 3      # mother must be an established track of >= this many nodes back (1 = off)
CHAIN_CAP        = 12     # cap on chain-length walk (cheap safety bound)

# --- eval ---
N_VAL        = 20
EVAL_N_VAL   = 5          # Loop A: comparability vs v2 (edge-J OFF vs ON) on the last-N val
DIV_RICH_N   = 8          # Loop B: # division-RICHEST train samples (top-8 ~ 31 GT divisions -> firm div-J denom)
MATCH_UM     = 7.0        # node-match gate (edge + division metric)
DIV_DT_TOL   = 1          # +-1 frame tolerance for division matching (per the metric spec)
EVAL_WEIGHTS = None

In [11]:
# ================= IO (zarr, same as v1/v2) =================
def open_image(zarr_path):
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g); return best

def load_geff(geff_path):
    g = zarr.open(geff_path, mode='r')
    node_ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g[f'nodes/props/{k}/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0, 2), dtype=np.uint64)
    return node_ids, props, edges

def gt_nodes_edges(geff_path):
    node_ids, props, edges = load_geff(geff_path)
    nodes = [(int(node_ids[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
             for i in range(len(node_ids))]
    return nodes, [(int(s), int(t)) for s, t in edges]

def build_sample_list(train_dir):
    out = []
    for gp in sorted(glob.glob(os.path.join(train_dir, '*.geff'))):
        zp = gp[:-5] + '.zarr'
        if os.path.exists(zp): out.append((zp, gp))
    return out

In [12]:
# ================= MODEL: anisotropic 3D U-Net (verbatim from v1) =================
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True))
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)

def find_weights():
    if EVAL_WEIGHTS: return EVAL_WEIGHTS
    for pat in ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt', '*.pt'):
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits: return hits[0]
    return None

def extract_state(ck):
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

In [13]:
# ================= DETECTION: full-res UNet heatmap -> peaks -> physical NMS =================
def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)

@torch.no_grad()
def predict_heatmap(model, vol):
    v = _normalize(vol)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def physical_nms(coords, scores, radius_um, scale=VOXEL):
    """Greedy non-max suppression in PHYSICAL (um) space: keep the strongest peak, drop others
    within radius_um. Isotropic in real units, unlike a voxel-space min_distance."""
    if len(coords) <= 1 or not radius_um:
        return coords, scores
    pts = coords * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    killed = np.zeros(len(coords), bool); keep = []
    for i in order:
        if killed[i]: continue
        keep.append(int(i))
        killed[tree.query_ball_point(pts[i], r=radius_um)] = True   # suppresses i itself + neighbours
    keep = np.array(keep)
    return coords[keep], scores[keep]

def detect_frames(model, arr, thr=HM_THR, min_distance=MIN_DIST, max_peaks=MAX_PEAKS,
                  nms_um=NMS_UM, refine=REFINE, win=REFINE_WIN):
    """Per-frame: heatmap -> peak_local_max -> (refine) -> physical NMS. Returns list of (N,3) coords."""
    model.eval(); frames = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32)
        hm = predict_heatmap(model, vol)
        pk = peak_local_max(hm, min_distance=min_distance, threshold_abs=thr,
                            exclude_border=False, num_peaks=max_peaks)
        if len(pk) == 0:
            frames.append(np.zeros((0, 3))); continue
        scores = hm[pk[:, 0], pk[:, 1], pk[:, 2]].astype(float)
        coords = pk.astype(np.float64)
        if refine:
            coords = refine_centroids(vol, coords, win=win)
        coords, scores = physical_nms(coords, scores, nms_um)
        frames.append(coords)
    return frames

In [14]:
# ================= LINKING + POST-PROCESSING (rule-based v2 stack) =================
@dataclass
class TrackGraph:
    node_t: np.ndarray; node_z: np.ndarray; node_y: np.ndarray; node_x: np.ndarray
    node_ids: np.ndarray; edges: np.ndarray; meta: dict
    @property
    def n_nodes(self): return len(self.node_ids)
    @property
    def n_edges(self): return len(self.edges)

def refine_centroids(vol, coords, win=(1, 3, 3)):
    if len(coords) == 0:
        return coords
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z - wz), min(Z, z + wz + 1)
        y0, y1 = max(0, y - wy), min(Y, y + wy + 1)
        x0, x1 = max(0, x - wx), min(X, x + wx + 1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (patch * zz).sum() / s
        out[i, 1] = (patch * yy).sum() / s
        out[i, 2] = (patch * xx).sum() / s
    return out

def link_twopass(frames, tight_um=6.0, loose_um=8.0, vel_blend=0.5):
    node_ids = []; node_t = []; node_z = []; node_y = []; node_x = []; frame_ids = []; nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for (z, y, x) in coords:
            node_ids.append(nid); node_t.append(t); node_z.append(z); node_y.append(y); node_x.append(x)
            ids.append(nid); nid += 1
        frame_ids.append(ids)
    def _hun(P, C, pred, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0: return []
        BIG = 1e9
        Draw = np.sqrt(((P[pi][:, None] - C[ci][None]) ** 2).sum(2))
        D = np.sqrt(((pred[pi][:, None] - C[ci][None]) ** 2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]
    edges = []; prev_xyz = np.zeros((0, 3)); prev_ids = []; prev_vel = None
    for t in range(len(frames)):
        curr = np.asarray(frames[t], float).reshape(-1, 3); curr_ids = frame_ids[t]
        if t > 0 and len(prev_ids) and len(curr):
            P = prev_xyz * SCALE[None, :]; C = curr * SCALE[None, :]
            pred = P + (vel_blend * prev_vel if prev_vel is not None else 0.0)
            N, M = len(P), len(C)
            links = _hun(P, C, pred, np.arange(N), np.arange(M), min(tight_um, loose_um))
            up = {p for p, _ in links}; uc = {c for _, c in links}
            fp = np.array([i for i in range(N) if i not in up], int)
            fc = np.array([j for j in range(M) if j not in uc], int)
            links += _hun(P, C, pred, fp, fc, loose_um)
            vel = np.zeros((N, 3)); nv = np.zeros((M, 3))
            for p, c in links:
                edges.append((prev_ids[p], curr_ids[c])); vel[p] = (curr[c] - prev_xyz[p]) * SCALE
            for p, c in links:
                nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_xyz, prev_ids = curr, curr_ids
    return TrackGraph(node_t=np.array(node_t, np.int64), node_z=np.array(node_z, float),
                      node_y=np.array(node_y, float), node_x=np.array(node_x, float),
                      node_ids=np.array(node_ids, np.int64),
                      edges=np.array(edges, np.int64).reshape(-1, 2), meta={})

def close_gaps(frames, g, max_gap=1, gap_dist_um=8.0):
    if g.n_edges == 0:
        return g
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    ends_by_t = {}; starts_by_t = {}
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out: ends_by_t.setdefault(t, []).append(nid)
        if nid not in has_in: starts_by_t.setdefault(t, []).append(nid)
    new_nodes = []; new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1)
            if not starts: continue
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :]) ** 2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            big = thr * 1000 + 1
            cost = np.where(d <= thr, d, big)
            ri, ci = linear_sum_assignment(cost)
            used_s = set()
            for r, c in zip(ri, ci):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s: continue
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]; ts, zs, ys, xs = coords[s_id]
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac; yi = ye + (ys - ye) * frac; xi = xe + (xs - xe) * frac
                    nid = next_id; next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid)); new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id)); has_out.add(e_id); used_s.add(c)
    if not new_nodes:
        return g
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g):
    if g.n_edges == 0: return g
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    if len(keep) == len(g.node_ids): return g
    return TrackGraph(node_t=g.node_t[keep], node_z=g.node_z[keep], node_y=g.node_y[keep],
                      node_x=g.node_x[keep], node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta)

def filter_short_tracks(g, min_len):
    if g.n_edges == 0 or min_len <= 1: return g
    parent = {int(n): int(n) for n in g.node_ids}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s, t in g.edges.reshape(-1, 2):
        ra, rb = find(int(s)), find(int(t))
        if ra != rb: parent[ra] = rb
    comp = {}
    for n in g.node_ids:
        comp.setdefault(find(int(n)), []).append(int(n))
    keep = set()
    for members in comp.values():
        if len(members) >= min_len: keep.update(members)
    idx = [i for i, n in enumerate(g.node_ids) if int(n) in keep]
    keepset = set(int(g.node_ids[i]) for i in idx)
    edges = np.array([(int(s), int(t)) for s, t in g.edges.reshape(-1, 2)
                      if int(s) in keepset and int(t) in keepset], dtype=np.int64).reshape(-1, 2)
    return TrackGraph(node_t=g.node_t[idx], node_z=g.node_z[idx], node_y=g.node_y[idx],
                      node_x=g.node_x[idx], node_ids=g.node_ids[idx], edges=edges, meta=g.meta)

def graph_to_nodes_edges(g):
    nodes = [(int(g.node_ids[i]), int(g.node_t[i]), int(round(g.node_z[i])),
              int(round(g.node_y[i])), int(round(g.node_x[i]))) for i in range(g.n_nodes)]
    edges = [(int(s), int(t)) for s, t in g.edges]
    return nodes, edges

In [15]:
# ================= METRIC: official edge-Jaccard (copied from v1) =================
def match_nodes_per_t(gt_nodes, pred_nodes, max_um=MATCH_UM):
    gt_by_t, pr_by_t = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by_t[n[1]].append(n)
    for n in pred_nodes: pr_by_t[n[1]].append(n)
    g2p = {}
    for t, G in gt_by_t.items():
        P = pr_by_t.get(t, [])
        if not P: continue
        A = np.array([[g[2], g[3], g[4]] for g in G]) * VOXEL
        B = np.array([[p[2], p[3], p[4]] for p in P]) * VOXEL
        D = cdist(A, B); r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, max_um=MATCH_UM):
    g2p = match_nodes_per_t(gt_nodes, pred_nodes, max_um)
    p2g = {v: k for k, v in g2p.items()}
    pred_set, gt_set = set(map(tuple, pred_edges)), set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:
            TP += (p2g[a], p2g[b]) in gt_set; FP += (p2g[a], p2g[b]) not in gt_set
    FN = sum(1 for u, v in gt_edges if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    d = TP + FP + FN
    return dict(jaccard=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN,
                matched_gt=len(g2p), n_gt_nodes=len(gt_nodes), n_pred_nodes=len(pred_nodes))

In [16]:
# ================= NEW: post-hoc DIVISION detection + division metric =================
from collections import Counter

def _adjacency(g):
    """coords{nid:(t,z,y,x)}, out_map{nid:[children]}, in_map{nid:[parents]} for the current graph."""
    coords = {int(g.node_ids[i]): (int(g.node_t[i]), float(g.node_z[i]), float(g.node_y[i]), float(g.node_x[i]))
              for i in range(g.n_nodes)}
    out_map, in_map = defaultdict(list), defaultdict(list)
    for s, t in g.edges.reshape(-1, 2):
        out_map[int(s)].append(int(t)); in_map[int(t)].append(int(s))
    return coords, out_map, in_map

def _walk_len(start, nbr_map, cap=CHAIN_CAP):
    """Length (#nodes incl. start) of the chain following the first neighbour each step, capped."""
    n, cur, seen = 1, start, {start}
    while nbr_map.get(cur):
        nxt = nbr_map[cur][0]
        if nxt in seen: break
        seen.add(nxt); cur = nxt; n += 1
        if n >= cap: break
    return n

def detect_divisions(g, radius_um=DIV_RADIUS_UM, min_angle_deg=MIN_ANGLE_DEG,
                     min_daughter_len=MIN_DAUGHTER_LEN, min_mother_len=MIN_MOTHER_LEN, scale=SCALE):
    """Add mother->second-daughter edges (out-degree 1 -> 2) where geometry + persistence gates pass.
    Returns (new_graph, n_divisions_added). Only reads the 1:1 linked graph; adds edges, never nodes."""
    if not DIV_ENABLE or g.n_edges == 0:
        return g, 0
    coords, out_map, in_map = _adjacency(g)
    by_t = defaultdict(list)
    for nid, (t, z, y, x) in coords.items():
        by_t[t].append(nid)
    cos_thr = np.cos(np.deg2rad(min_angle_deg))   # require cos(angle) <= cos_thr  => angle >= min_angle
    new_edges = []; claimed = set()
    for m in sorted(coords):
        outs = out_map.get(m, [])
        if len(outs) != 1:                        # extend only established 1:1 continuations
            continue
        if min_mother_len > 1 and _walk_len(m, in_map) < min_mother_len:
            continue
        d1 = outs[0]
        tm, zm, ym, xm = coords[m]
        t1, z1, y1, x1 = coords[d1]
        pm = np.array([zm, ym, xm]); pd1 = np.array([z1, y1, x1])
        a = (pd1 - pm) * scale; na = np.linalg.norm(a)
        if na < 1e-6:
            continue
        best = None
        for c in by_t.get(t1, []):                # candidate second daughters live at the daughter frame
            if c == d1 or c in claimed or in_map.get(c):   # must be an ORPHAN birth (no parent yet)
                continue
            tc, zc, yc, xc = coords[c]
            pc = np.array([zc, yc, xc])
            b = (pc - pm) * scale; nb = np.linalg.norm(b)
            if nb > radius_um or nb < 1e-6:
                continue
            if float(np.dot(a, b) / (na * nb)) > cos_thr:   # angle too small (daughters not opposite)
                continue
            if _walk_len(c, out_map) < min_daughter_len:    # orphan must persist -> real daughter
                continue
            if best is None or nb < best[0]:
                best = (nb, c)
        if best is not None:
            new_edges.append((m, best[1])); claimed.add(best[1])
    if not new_edges:
        return g, 0
    edges = np.concatenate([g.edges.reshape(-1, 2), np.array(new_edges, np.int64).reshape(-1, 2)])
    g2 = TrackGraph(node_t=g.node_t, node_z=g.node_z, node_y=g.node_y, node_x=g.node_x,
                    node_ids=g.node_ids, edges=edges, meta=g.meta)
    return g2, len(new_edges)

def division_events(nodes, edges):
    """Return [(t,z,y,x)] for every node with out-degree >= 2 (a division mother)."""
    out_deg = Counter(int(s) for s, _ in edges)
    coord = {int(nid): (t, z, y, x) for (nid, t, z, y, x) in nodes}
    return [coord[n] for n, d in out_deg.items() if d >= 2 and n in coord]

def division_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, max_um=MATCH_UM, dt_tol=DIV_DT_TOL):
    """Match predicted division mothers to GT division mothers (Hungarian, physical gate + -+dt_tol frames).
    Returns jaccard=TP/(TP+FP+FN) over division EVENTS. NaN when there are no divisions on either side."""
    G = division_events(gt_nodes, gt_edges)
    P = division_events(pred_nodes, pred_edges)
    TP = 0
    if G and P:
        A = np.array([[g[1], g[2], g[3]] for g in G]) * VOXEL
        B = np.array([[p[1], p[2], p[3]] for p in P]) * VOXEL
        tA = np.array([g[0] for g in G]); tB = np.array([p[0] for p in P])
        D = cdist(A, B); dt = np.abs(tA[:, None] - tB[None, :])
        BIG = 1e9
        cost = np.where((D <= max_um) & (dt <= dt_tol), D, BIG)
        r, c = linear_sum_assignment(cost)
        TP = int(sum(1 for i, j in zip(r, c) if cost[i, j] < BIG))
    FP = len(P) - TP; FN = len(G) - TP
    denom = TP + FP + FN
    return dict(jaccard=(TP / denom if denom else float('nan')), TP=TP, FP=FP, FN=FN,
                n_gt_div=len(G), n_pred_div=len(P))

def count_gt_divisions(geff_path):
    """Cheap: #GT division mothers (out-degree >= 2) from edges only. Used to rank division-rich samples."""
    _, _, edges = load_geff(geff_path)
    if len(edges) == 0: return 0
    return sum(1 for _, d in Counter(int(s) for s, _ in edges).items() if d >= 2)

In [ ]:
# ================= RUN: v2 (LB-best) pipeline + post-hoc divisions -> edge-J (OFF/ON) + division-J =================
model = UNet3D(base=BASE, strides=STRIDES).to(device)
wp = find_weights()
assert wp, 'no .pt found under /kaggle/input -- add your weights dataset (v1_UNet_best.pt) as an input.'
model.load_state_dict(extract_state(torch.load(wp, map_location=device))); model.eval()
print('loaded weights:', wp)
print(f'detect: HM_THR={HM_THR} min_dist={MIN_DIST} NMS_UM={NMS_UM} | link: loose={LOOSE_UM} filter={FILTER_MIN_LEN}')
print(f'division gates: radius={DIV_RADIUS_UM}um angle>={MIN_ANGLE_DEG} daughter_len>={MIN_DAUGHTER_LEN} '
      f'mother_len>={MIN_MOTHER_LEN}\n')

def track_linked(model, arr):
    """v2 (LB-best): detect (HM_THR 0.3, no NMS) -> two-pass link -> close_gaps -> prune -> filter. Returns the 1:1 graph."""
    frames = detect_frames(model, arr)
    g = link_twopass(frames, tight_um=TIGHT_UM, loose_um=LOOSE_UM, vel_blend=VEL_BLEND)
    if CLOSE_GAPS:     g = close_gaps(frames, g, max_gap=MAX_GAP, gap_dist_um=GAP_DIST_UM)
    if PRUNE_ISOLATED: g = prune_isolated(g)
    if FILTER_MIN_LEN > 1: g = filter_short_tracks(g, FILTER_MIN_LEN)
    return g

samples = build_sample_list(TRAIN_DIR)

# ---------- Loop A: edge-J safety on the same last-EVAL_N_VAL val as v2/v2.5 (OFF vs ON) ----------
print(f"=== Loop A: edge-J OFF vs ON, last-{EVAL_N_VAL} val (v2 baseline 0.859, FP~0) ===")
val_s = samples[-N_VAL:][:EVAL_N_VAL]
oTP=oFP=oFN=0; nTP=nFP=nFN=0; dTP=dFP=dFN=0; added=0; t0=time.time()
for zp, gp in val_s:
    g = track_linked(model, open_image(zp))
    gt_nodes, gt_edges = gt_nodes_edges(gp)
    n0, e0 = graph_to_nodes_edges(g)                       # divisions OFF
    r0 = edge_jaccard(gt_nodes, gt_edges, n0, e0)
    gd, k = detect_divisions(g); added += k                # divisions ON
    n1, e1 = graph_to_nodes_edges(gd)
    r1 = edge_jaccard(gt_nodes, gt_edges, n1, e1)
    dj = division_jaccard(gt_nodes, gt_edges, n1, e1)
    oTP+=r0['TP']; oFP+=r0['FP']; oFN+=r0['FN']
    nTP+=r1['TP']; nFP+=r1['FP']; nFN+=r1['FN']
    dTP+=dj['TP']; dFP+=dj['FP']; dFN+=dj['FN']
    print(f"  {os.path.basename(gp)[:-5]}: edgeJ {r0['jaccard']:.3f}->{r1['jaccard']:.3f} "
          f"(FP {r0['FP']}->{r1['FP']}) | +{k} div | GTdiv={dj['n_gt_div']} predDiv={dj['n_pred_div']}")
jo = oTP/(oTP+oFP+oFN+1e-9); jn = nTP/(nTP+nFP+nFN+1e-9)
print(f"\nedge-J  OFF={jo:.4f} (FP={oFP})  ON={jn:.4f} (FP={nFP})   [delta={jn-jo:+.4f}; want ~0 = divisions don't hurt]")
print(f"divisions added: {added} | division-J on this val: TP={dTP} FP={dFP} FN={dFN} "
      f"(often ~0 GT divisions here -> see Loop B)  [{time.time()-t0:.0f}s]")

# ---------- Loop B: division-J where GT divisions actually exist (division-richest train samples) ----------
print(f"\n=== Loop B: division-J on the {DIV_RICH_N} division-RICHEST train samples ===")
ranked = sorted(((count_gt_divisions(gp), zp, gp) for zp, gp in samples), key=lambda x: -x[0])
rich = [(zp, gp) for nd, zp, gp in ranked if nd > 0][:DIV_RICH_N]
if not rich:
    print("  no train sample has a GT division -> the 0.1 term is unreachable; skip divisions.")
else:
    bTP=bFP=bFN=0; beTPo=beFNo=beTPn=beFPn=beFNn=0; badded=0; t0=time.time()
    for zp, gp in rich:
        g = track_linked(model, open_image(zp))
        gt_nodes, gt_edges = gt_nodes_edges(gp)
        e_off = edge_jaccard(gt_nodes, gt_edges, *graph_to_nodes_edges(g))
        gd, k = detect_divisions(g); badded += k
        n1, e1 = graph_to_nodes_edges(gd)
        e_on = edge_jaccard(gt_nodes, gt_edges, n1, e1)
        dj = division_jaccard(gt_nodes, gt_edges, n1, e1)
        bTP+=dj['TP']; bFP+=dj['FP']; bFN+=dj['FN']
        beTPo+=e_off['TP']; beFNo+=e_off['FN']; beTPn+=e_on['TP']; beFPn+=e_on['FP']; beFNn+=e_on['FN']
        print(f"  {os.path.basename(gp)[:-5]}: GTdiv={dj['n_gt_div']} predDiv={dj['n_pred_div']} "
              f"divTP={dj['TP']} divFP={dj['FP']} divFN={dj['FN']} | edgeJ {e_off['jaccard']:.3f}->{e_on['jaccard']:.3f} "
              f"(edgeFP {e_off['FP']}->{e_on['FP']}) +{k}div")
    denom = bTP+bFP+bFN
    div_j = bTP/denom if denom else float('nan')
    ejo = beTPo/(beTPo+0+beFNo+1e-9); ejn = beTPn/(beTPn+beFPn+beFNn+1e-9)
    print(f"\nmicro division-J: {div_j:.4f} (TP={bTP} FP={bFP} FN={bFN}) over {badded} predicted divisions")
    print(f"edge-J on these:  OFF={ejo:.4f} -> ON={ejn:.4f} (edge-FP total ON={beFPn})")
    print(f"=> contribution to the competition score: 0.1 * {div_j:.4f} = {0.1*div_j:+.4f} "
          f"(if edge-J holds)  [{time.time()-t0:.0f}s]")
print("\nbaseline: v2 edge-J 0.859 (LB 0.827), division-J 0.0 (Hungarian is 1:1 -> no divisions).")

**How to read it.**

- **Loop A is the safety check.** `edge-J OFF vs ON` on the same last-5 val as v2 (baseline 0.859). The
  delta should be **~0** and `FP` should barely move — that confirms the added `M→D2` edges land on orphan
  detections and are *not* charged as edge FP, so divisions don't threaten the 0.827 main score. If edge-J
  *drops* or FP jumps, the gates are too loose (raise `MIN_DAUGHTER_LEN` / `MIN_ANGLE_DEG`, or lower
  `DIV_RADIUS_UM`) — a division that steals a real link is exactly the LB-hurting FP we must avoid.

- **Loop B is the signal.** On the division-richest samples, read **micro division-J** and its score
  contribution `0.1 * div-J`. `divFP` = false divisions (predicted where there is no GT division) — these
  directly lower div-J, so if `divFP ≫ divTP`, tighten the gates for **precision** (divisions are rare, so
  precision matters more than recall here). `divFN` = real divisions missed (detection didn't find the
  second daughter, a gate rejected it, or `close_gaps` had already claimed the orphan).

**Tuning order (one gate at a time, re-run RUN):**
1. `MIN_DAUGHTER_LEN` — 2 / 3 / 4. Higher = fewer false divisions from transient orphans (biggest precision lever). Lower to 2 if div-recall is too low.
2. `MIN_ANGLE_DEG` — 60 / 75 / 90. GT split-angle p10=88° → 75 keeps nearly all real splits; raise toward 90 only if same-side FPs appear.
3. `DIV_RADIUS_UM` — 8 / 10 / 12. GT mother→daughter p90=9.07, max=13.5 → 10 ≈ p90; 12 chases the tail (more recall, more FP).
4. `MIN_MOTHER_LEN` — 1 (off) / 3.

**Decision rule.** If Loop A holds (edge-J flat, FP flat) AND Loop B shows a clearly positive div-J with
`divFP` under control → port `detect_divisions` + its config into `src/submit.ipynb` (call it once after
`filter_short_tracks`, before writing edges) and submit as a **single isolated change** vs the current
v2.5b/v2 config. The LB is the ground truth for divisions on the hidden set; a good local div-J is
permission to spend one submission, not proof. If Loop A regresses, do NOT submit — retune gates first.